In [1]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt

from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2

df = pd.read_csv("Womens Clothing E-Commerce Reviews.csv")

print("Dataset Shape:", df.shape)
print("\nRequired Columns:")
print(df[["Review Text", "Recommended IND"]].head())

print("\nMissing Values:")
print(df[["Review Text", "Recommended IND"]].isnull().sum())

Dataset Shape: (23486, 11)

Required Columns:
                                         Review Text  Recommended IND
0  Absolutely wonderful - silky and sexy and comf...                1
1  Love this dress!  it's sooo pretty.  i happene...                1
2  I had such high hopes for this dress and reall...                0
3  I love, love, love this jumpsuit. it's fun, fl...                1
4  This shirt is very flattering to all due to th...                1

Missing Values:
Review Text        845
Recommended IND      0
dtype: int64


In [2]:
df_text = df[["Review Text", "Recommended IND"]].copy()

df_text = df_text.dropna(subset=["Review Text"])

df_text["Review Text"] = df_text["Review Text"].astype(str).str.strip()

df_text = df_text[df_text["Review Text"] != ""].reset_index(drop=True)

print("Original Rows:", len(df))
print("Remaining Rows:", len(df_text))
print("Removed Rows:", len(df) - len(df_text))

print("\nMissing Values:")
print(df_text.isnull().sum())

print("\nTarget Distribution:")
print(df_text["Recommended IND"].value_counts())

Original Rows: 23486
Remaining Rows: 22641
Removed Rows: 845

Missing Values:
Review Text        0
Recommended IND    0
dtype: int64

Target Distribution:
Recommended IND
1    18540
0     4101
Name: count, dtype: int64


In [3]:
nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

df_text["Clean Review"] = df_text["Review Text"].apply(preprocess_text)

df_text = df_text[df_text["Clean Review"] != ""].reset_index(drop=True)

print(df_text[["Review Text", "Clean Review"]].head())
print("\nFinal Rows:", len(df_text))

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\panch\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\panch\AppData\Roaming\nltk_data...


                                         Review Text  \
0  Absolutely wonderful - silky and sexy and comf...   
1  Love this dress!  it's sooo pretty.  i happene...   
2  I had such high hopes for this dress and reall...   
3  I love, love, love this jumpsuit. it's fun, fl...   
4  This shirt is very flattering to all due to th...   

                                        Clean Review  
0  absolutely wonderful silky and sexy and comfor...  
1  love this dress it s sooo pretty i happened to...  
2  i had such high hope for this dress and really...  
3  i love love love this jumpsuit it s fun flirty...  
4  this shirt is very flattering to all due to th...  

Final Rows: 22641


In [4]:
configurations = {
    "Unigram_No_Stopwords": {
        "ngram_range": (1, 1),
        "stop_words": None
    },
    "Unigram_With_Stopwords": {
        "ngram_range": (1, 1),
        "stop_words": "english"
    },
    "Unigram_Bigram_No_Stopwords": {
        "ngram_range": (1, 2),
        "stop_words": None
    },
    "Unigram_Bigram_With_Stopwords": {
        "ngram_range": (1, 2),
        "stop_words": "english"
    }
}

tfidf_results = {}
experiment_records = []

for name, settings in configurations.items():
    vectorizer = TfidfVectorizer(
        ngram_range=settings["ngram_range"],
        stop_words=settings["stop_words"],
        min_df=5,
        max_df=0.95
    )

    matrix = vectorizer.fit_transform(df_text["Clean Review"])

    tfidf_results[name] = {
        "vectorizer": vectorizer,
        "matrix": matrix
    }

    experiment_records.append({
        "Configuration": name,
        "Documents": matrix.shape[0],
        "Features": matrix.shape[1],
        "Non_Zero_Values": matrix.nnz,
        "Sparsity_Percentage": (
            1 - matrix.nnz / (matrix.shape[0] * matrix.shape[1])
        ) * 100
    })

experiment_df = pd.DataFrame(experiment_records)

print(experiment_df)

                   Configuration  Documents  Features  Non_Zero_Values  \
0           Unigram_No_Stopwords      22641      4294           932548   
1         Unigram_With_Stopwords      22641      4037           512369   
2    Unigram_Bigram_No_Stopwords      22641     33772          1864175   
3  Unigram_Bigram_With_Stopwords      22641     21323           794204   

   Sparsity_Percentage  
0            99.040790  
1            99.439432  
2            99.756200  
3            99.835492  


In [5]:
y = df_text["Recommended IND"]

chi_square_results = {}
comparison_records = []

for name, result in tfidf_results.items():
    matrix = result["matrix"]
    vectorizer = result["vectorizer"]

    scores, p_values = chi2(matrix, y)
    feature_names = vectorizer.get_feature_names_out()

    score_df = pd.DataFrame({
        "Feature": feature_names,
        "Chi_Square_Score": scores,
        "P_Value": p_values
    })

    score_df = score_df.sort_values(
        "Chi_Square_Score",
        ascending=False
    ).reset_index(drop=True)

    chi_square_results[name] = score_df

    top_count = min(100, len(score_df))

    comparison_records.append({
        "Configuration": name,
        "Total_Features": len(score_df),
        "Top_100_Average_Score": score_df.head(top_count)[
            "Chi_Square_Score"
        ].mean(),
        "Highest_Chi_Square_Score": score_df[
            "Chi_Square_Score"
        ].max()
    })

chi_square_comparison = pd.DataFrame(comparison_records)

chi_square_comparison = chi_square_comparison.sort_values(
    "Top_100_Average_Score",
    ascending=False
).reset_index(drop=True)

print(chi_square_comparison)

                   Configuration  Total_Features  Top_100_Average_Score  \
0         Unigram_With_Stopwords            4037              40.130252   
1           Unigram_No_Stopwords            4294              35.734101   
2  Unigram_Bigram_With_Stopwords           21323              33.200904   
3    Unigram_Bigram_No_Stopwords           33772              29.235966   

   Highest_Chi_Square_Score  
0                177.863008  
1                148.666029  
2                120.944429  
3                 85.305411  


In [6]:
best_configuration = chi_square_comparison.iloc[0]["Configuration"]

best_vectorizer = tfidf_results[best_configuration]["vectorizer"]
best_matrix = tfidf_results[best_configuration]["matrix"]
best_score_df = chi_square_results[best_configuration].copy()

best_score_df["Chi_Square_Score"] = best_score_df[
    "Chi_Square_Score"
].fillna(0)

total_score = best_score_df["Chi_Square_Score"].sum()

best_score_df["Cumulative_Score_Ratio"] = (
    best_score_df["Chi_Square_Score"].cumsum() / total_score
)

optimal_k = (
    np.searchsorted(
        best_score_df["Cumulative_Score_Ratio"].values,
        0.80
    ) + 1
)

selector = SelectKBest(score_func=chi2, k=optimal_k)

selected_matrix = selector.fit_transform(
    best_matrix,
    y
)

all_feature_names = best_vectorizer.get_feature_names_out()

selected_feature_names = all_feature_names[
    selector.get_support()
]

selected_features_df = best_score_df.head(optimal_k).copy()

print("Best Configuration:", best_configuration)
print("Original Feature Count:", best_matrix.shape[1])
print("Selected Feature Count:", selected_matrix.shape[1])
print("Selected Matrix Shape:", selected_matrix.shape)

print("\nTop 30 Selected Features:")
print(
    selected_features_df[
        ["Feature", "Chi_Square_Score", "P_Value"]
    ].head(30)
)

Best Configuration: Unigram_With_Stopwords
Original Feature Count: 4037
Selected Feature Count: 629
Selected Matrix Shape: (22641, 629)

Top 30 Selected Features:
           Feature  Chi_Square_Score       P_Value
0               wa        177.863008  1.419147e-40
1     disappointed        168.398587  1.655543e-38
2     unflattering        136.486511  1.561709e-31
3            cheap        133.489427  7.065800e-31
4         returned        124.225640  7.518847e-29
5        returning        120.717745  4.405582e-28
6           looked        113.784830  1.452444e-26
7             huge        110.736802  6.757187e-26
8    unfortunately        104.074607  1.948394e-24
9           wanted         93.527639  4.005676e-22
10            poor         84.802628  3.296860e-20
11          return         71.905767  2.257233e-17
12           sadly         71.231809  3.176258e-17
13           awful         70.529865  4.533513e-17
14         excited         66.064810  4.363370e-16
15         perfect   

In [7]:
original_feature_count = best_matrix.shape[1]
selected_feature_count = selected_matrix.shape[1]

reduction_percentage = (
    1 - selected_feature_count / original_feature_count
) * 100

top_features = selected_features_df["Feature"].head(20).tolist()

print("FINAL TEXT VECTORIZATION RESULT")
print("-" * 45)
print("Vectorization Method: TF-IDF")
print("Best Configuration:", best_configuration)
print("Text Processing: Cleaning + Lemmatization")
print("Feature Selection Method: Chi-square")
print("Target Column: Recommended IND")
print("Original Features:", original_feature_count)
print("Selected Features:", selected_feature_count)
print(f"Feature Reduction: {reduction_percentage:.2f}%")
print("Final Matrix Shape:", selected_matrix.shape)

print("\nTop 20 Important Features:")
for position, feature in enumerate(top_features, start=1):
    print(position, feature)

FINAL TEXT VECTORIZATION RESULT
---------------------------------------------
Vectorization Method: TF-IDF
Best Configuration: Unigram_With_Stopwords
Text Processing: Cleaning + Lemmatization
Feature Selection Method: Chi-square
Target Column: Recommended IND
Original Features: 4037
Selected Features: 629
Feature Reduction: 84.42%
Final Matrix Shape: (22641, 629)

Top 20 Important Features:
1 wa
2 disappointed
3 unflattering
4 cheap
5 returned
6 returning
7 looked
8 huge
9 unfortunately
10 wanted
11 poor
12 return
13 sadly
14 awful
15 excited
16 perfect
17 comfortable
18 great
19 bad
20 love
